In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
# Imports & load data
import pandas as pd
import numpy as np

# point to your filtered file
file_path = '/content/drive/MyDrive/MRP/news_textrank_with_sentiment.csv'

# read, parsing the Date column as datetime
df = pd.read_csv(file_path, parse_dates=['Date'])

In [3]:
df.head()

,Date,Stock_symbol,Textrank_summary,sentiment_label,sentiment_score,sentiment_numeric,has_article
0,2016-01-06 00:00:00+00:00,A,Deutsche Bank Initiates Coverage on Agilent Te...,Neutral,0.998652,0,1
1,2016-01-06 00:00:00+00:00,A,Agilent to pay Enzo Bio $9M to settle patent f...,Neutral,0.999939,0,1
2,2016-01-07 00:00:00+00:00,A,Benzinga's Top Upgrades,Neutral,0.984295,0,1
3,2016-01-07 00:00:00+00:00,A,Cowen &amp; Company Upgrades Agilent Technolog...,Positive,0.999998,1,1
4,2016-02-05 00:00:00+00:00,A,Tracking William Von Mueffling's Cantillon Cap...,Neutral,0.997771,0,1


In [4]:
# Rename Date -> date and Stock_symbol -> symbol
df.rename(columns={
    'Date': 'date',
    'Stock_symbol': 'symbol'
}, inplace=True)

# Confirm renaming
df.columns

Index(['date', 'symbol', 'Textrank_summary', 'sentiment_label',
       'sentiment_score', 'sentiment_numeric', 'has_article'],
      dtype='object')

In [6]:
# Convert date column to date-only format (drop time)
df['date'] = pd.to_datetime(df['date']).dt.date

In [7]:
# Select only useful columns for modeling
df = df[['date', 'symbol', 'sentiment_numeric', 'sentiment_score', 'has_article']]

# Preview final format
df.head()

,date,symbol,sentiment_numeric,sentiment_score,has_article
0,2016-01-06,A,0,0.998652,1
1,2016-01-06,A,0,0.999939,1
2,2016-01-07,A,0,0.984295,1
3,2016-01-07,A,1,0.999998,1
4,2016-02-05,A,0,0.997771,1


In [8]:
# Aggregate multiple news entries per stock per day
df_news_agg = df.groupby(['date', 'symbol'], as_index=False).agg({
    'sentiment_numeric': 'mean',
    'sentiment_score': 'mean',
    'has_article': 'sum'  # counts how many news articles exist per date-symbol
})

In [10]:
df_news_agg.rename(columns={
    'sentiment_numeric': 'avg_sentiment',
    'sentiment_score': 'avg_sentiment_score',
    'has_article': 'news_count'
}, inplace=True)

df_news_agg.head(50)

,date,symbol,avg_sentiment,avg_sentiment_score,news_count
0,2016-01-01,ABBV,1.0,0.999724,1
1,2016-01-01,ACP,0.0,0.999792,1
2,2016-01-01,AIF,0.0,0.999792,1
3,2016-01-01,AKS,0.0,0.999909,1
4,2016-01-01,AMGN,1.0,0.999946,1
5,2016-01-01,AN,1.0,0.999958,1
6,2016-01-01,ANGL,0.0,0.999792,1
7,2016-01-01,ARDC,0.0,0.999792,1
8,2016-01-01,ASHR,-1.0,0.996545,1
9,2016-01-01,ATRO,-1.0,0.999999,1


In [11]:
# Sort the aggregated DataFrame by symbol and then by date
df_news_agg.sort_values(by=['symbol', 'date'], inplace=True)


In [12]:
df_news_agg.head()

,date,symbol,avg_sentiment,avg_sentiment_score,news_count
2694,2016-01-06,A,0.0,0.999295,2
3720,2016-01-07,A,0.5,0.992147,2
26786,2016-02-05,A,0.0,0.997771,1
30631,2016-02-10,A,0.0,0.999134,1
34333,2016-02-15,A,0.0,0.999934,1


In [13]:
# Save the filtered and renamed news dataset
output_path = '/content/drive/MyDrive/MRP/news_preprocessed.csv'
df_news_agg.to_csv(output_path, index=False)
print(f"Preprocessed news dataset saved to: {output_path}")

Preprocessed news dataset saved to: /content/drive/MyDrive/MRP/news_preprocessed.csv
